In [ ]:
import re
import shutil
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


In [ ]:
DATA_FOLDER = Path("../data")
FILE_PATH = DATA_FOLDER / "input-data.csv"
IDA_TEMPLATE_PATH = DATA_FOLDER / "template.dwdtry"
input_data = pd.read_csv(FILE_PATH.as_posix(), sep=";")
input_data.head()

In [ ]:
input_data.describe()

In [ ]:
def substitute_outliers(df:pd.DataFrame, column_name: str, outlier: Any):
    _df = df.copy()
    _df[column_name] = _df[column_name].replace(outlier, np.nan).ffill()
    return _df

In [ ]:
# Nous nettoyons les valeurs abérantes.

data = substitute_outliers(input_data, 'WR Windrichtung in 10 m Hoehe ueber Grund [Grad]', -999.0)
data = substitute_outliers(data, 'WG Windgeschwindigkeit in 10 m Hoehe ueber Grund [m/s]', -99.9)
data = substitute_outliers(data, 'N  Bedeckungsgrad [Achtel]', -1)
data = substitute_outliers(data, 'D  Diffuse Sonnenbetrahlungsstaerke (horiz. Ebene) [W/m�]', -360)

data.describe()

In [ ]:
# Make a copy of the IDA template file
OUTPUT_FILE = "../output/results.dwdtry"
shutil.copy(src=IDA_TEMPLATE_PATH.as_posix(), dst=OUTPUT_FILE)
print("File copied successfully.")

In [ ]:
DATA_FORMATER = "3904500 2510500  1  1  1   1.7 1001 131  4.4 7  3.9  88    0    0 278 -321  3"

def parse_format(template: str):
    """Extrait (largeur, nb_decimales) pour chaque champ du template."""
    specs = []
    for m in re.finditer(r"\s*\S+", template):
        token = m.group()
        width = len(token)                       # largeur totale, espaces inclus
        val = token.strip()
        decimals = len(val.split(".")[1]) if "." in val else 0
        specs.append((width, decimals))
    return specs

FORMAT_SPECS = parse_format(DATA_FORMATER)

def format_row(values, specs=FORMAT_SPECS):
    if len(values) != len(specs):
        raise ValueError(f"{len(values)} valeurs pour {len(specs)} champs")
    out = []
    for v, (w, d) in zip(values, specs):
        s = f"{v:>{w}.{d}f}"
        if len(s) > w:
            raise ValueError(f"Valeur {v} trop large pour un champ de {w} caractères")
        out.append(s)
    return "".join(out)

In [ ]:
with open(OUTPUT_FILE, mode="a") as f:
    f.writelines(format_row(row) + "\n" for row in data.itertuples(index=False))

print("Géneration du fichier terminé")